In [1]:
from plotly.io import show
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_factors_dataset, load_sp500_dataset
from skfolio.optimization import MeanRisk, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import BlackLitterman, FactorModel

prices = load_sp500_dataset()
factor_prices = load_factors_dataset()

prices = prices["2014":]
factor_prices = factor_prices["2014":]

X, y = prices_to_returns(prices, factor_prices)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, shuffle=False)

In [2]:
factor_views = [
    "SIZE == 0.00039",
    "SIZE - VLUE == 0.00011 ",
    "MTUM - QUAL == 0.00007",
]

In [3]:
model_bl_factor = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    prior_estimator=FactorModel(
        factor_prior_estimator=BlackLitterman(views=factor_views),
    ),
    portfolio_params=dict(name="Black & Litterman Factor Model"),
)
model_bl_factor.fit(X_train, y_train)
model_bl_factor.weights_

array([5.55998971e-02, 2.42678756e-02, 3.63991372e-02, 1.97989097e-02,
       4.38883396e-08, 1.69582846e-02, 1.34103109e-01, 3.52967041e-07,
       2.78013243e-02, 9.56133325e-02, 6.72490988e-02, 9.04221682e-02,
       1.24559355e-01, 8.98176230e-02, 5.85674115e-02, 2.83525179e-02,
       6.69595123e-03, 1.02316969e-01, 2.14765919e-02, 4.61288086e-08])

In [4]:
model_factor = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    prior_estimator=FactorModel(),
    portfolio_params=dict(name="Factor Model"),
)
model_factor.fit(X_train, y_train)
model_factor.weights_

array([1.03294288e-06, 1.27482685e-03, 4.19682803e-07, 3.34130824e-06,
       7.36838285e-07, 1.28824408e-06, 5.13031432e-02, 6.35619183e-02,
       6.14804832e-07, 1.79106051e-01, 5.03130911e-02, 7.13734379e-02,
       4.13002526e-02, 2.27978407e-01, 5.13348034e-02, 1.44130375e-01,
       2.99026115e-07, 6.19737850e-02, 5.63413085e-02, 8.67773195e-07])

In [5]:
ptf_bl_factor_test = model_bl_factor.predict(X_test)
ptf_factor_test = model_factor.predict(X_test)

population = Population([ptf_bl_factor_test, ptf_factor_test])

population.plot_cumulative_returns()

In [6]:
fig = population.plot_composition()
show(fig)

In [7]:
assets_views = [
    "AAPL == 0.00098",
    "AAPL - GE == 0.00086",
    "JPM - GE == 0.00059",
]

model = BlackLitterman(
    views=assets_views,
    prior_estimator=FactorModel(
        factor_prior_estimator=BlackLitterman(views=factor_views),
    ),
)

model.fit(X, y)
print(model.return_distribution_.covariance.shape)

(20, 20)
